# Classification Evaluator

Evaluates a model's **classification accuracy** on a labelled dataset.

For each row the evaluator:
- Sends the input text to the model as a chat prompt
- Compares the model's response to the ground-truth label
- Computes per-class Precision, Recall, F1 and macro/micro averages

**Supported data sources:** local files (`file:`), DBFS (`dbfs:`), Azure Data Lake (`adls:`).
This tutorial uses a local Parquet file.

## 1. Imports

In [ ]:
import pandas as pd
from bellmira.evaluators import ModelClassificationEvaluator

## 2. Prepare a sample dataset

Create a small Parquet file with an `input` column (text to classify) and a `label` column (ground truth).
Skip this cell if you already have a dataset.

In [ ]:
import os

sample_data = {
    "input": [
        "Preciso de ajuda para bloquear o meu cartão de débito.",
        "Quero fazer uma transferência para outra conta.",
        "Tenho dúvidas sobre o meu crédito habitação.",
        "Não consigo entrar na aplicação mobile do banco.",
        "Quero ativar o MBway no meu telemóvel.",
        "Como posso atualizar a minha morada?",
        "Preciso de informação sobre fundos de investimento.",
        "O meu cartão foi recusado numa compra.",
    ],
    "label": [
        "Cartões",
        "Transferências",
        "Crédito",
        "Site e Mobile",
        "MBway",
        "Atualização Dados",
        "Investimentos",
        "Cartões",
    ],
}

os.makedirs("/tmp/bellmira_tutorial", exist_ok=True)
df = pd.DataFrame(sample_data)
df.to_parquet("/tmp/bellmira_tutorial/classification_sample.parquet", index=False)
print(f"Dataset saved with {len(df)} rows.")
df.head()

## 3. Define the system prompt

The system prompt tells the model the list of valid classes and how to respond.
The example below is the actual call-center categorisation prompt used in production.

In [ ]:
SYSTEM_PROMPT = """
És um especialista de análise de conteúdos de chamadas de Call Center do Millennium BCP.

A tua função é categorizar a chamada com base no texto fornecido.

Lista de categorias:
* Transferências
* Cartões
* Crédito
* Investimentos
* Poupanças
* Contas
* Serviços Emergência
* Pagamentos
* MBway
* Atualização Dados
* Site e Mobile
* Seguros
* Outras

Devolve apenas a categoria. Não escrevas mais nada.
"""

## 4. Initialise the evaluator

In [ ]:
MODEL_URL = "https://chatqwen25-15-cc-cla-backend.dat-aip-millm.qa.mbcp.cloud/"

evaluator = ModelClassificationEvaluator(
    input_col="input",
    output_col="label",
    url=MODEL_URL,
    data_path="file:///tmp/bellmira_tutorial/classification_sample.parquet",
    data_format="parquet",
    system_prompt=SYSTEM_PROMPT,
    temperature=0.0,
)

## 5. Run evaluation

In [ ]:
# max_prompts limits how many rows are evaluated
results = evaluator.evaluate(max_prompts=8)

# Each item: Code, Execution_time, Prompt_tokens, Completion_tokens, Prediction, Label
for r in results:
    match = "✓" if r.get("Prediction") == r.get("Label") else "✗"
    print(f"  {match}  label={r.get('Label')!r:20s}  pred={r.get('Prediction')!r}")

## 6. Extract classification metrics

Returns per-class TP/FP/FN/TN, macro and micro Precision/Recall/F1, accuracy, and a confusion matrix.

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)

print(f"Accuracy:        {metrics['Accuracy']}")
print(f"Macro F1:        {metrics['Macro-F1']}")
print(f"Micro F1:        {metrics['Micro-F1']}")
print(f"Avg latency:     {metrics['Avg_Execution_Time']}s")
print(f"Avg prompt tok:  {metrics['Avg_Prompt_Tokens']}")
print(f"Avg compl  tok:  {metrics['Avg_Completion_Tokens']}")

## 7. Log to MLflow (optional)

In [ ]:
import mlflow

mlflow.set_experiment("classification_evaluation")
with mlflow.start_run(run_name="Qwen25-classification"):
    loggable = {k: v for k, v in metrics.items() if isinstance(v, (int, float)) and v is not None}
    mlflow.log_metrics(loggable)
    mlflow.log_param("model_url", MODEL_URL)
    print("Logged to MLflow:", loggable)